# 05 — Run structured lessons

This executes the structured-lesson memory condition on the same 12 tasks and same Vertex model as the baseline. Each completed answer is converted into a structured lesson. Only tasks 2 and 3 of each episode retrieve one prior lesson by relevance.

The run includes lesson-generation cost and latency in the primary totals, then records retrieval overhead separately.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
RESULTS_PATH = RUNS_DIR / 'structured_lesson_results.jsonl'
MEMORY_PATH = RUNS_DIR / 'structured_lessons.sqlite3'
RUNNER_PATH = EXPERIMENTS_DIR / 'run_vertex_structured_lessons.py'
if not MANIFEST_PATH.is_file():
    raise RuntimeError('Run 01_build_task_manifest.ipynb first.')
print('Vertex settings:', {key: os.environ.get(key, default) for key, default in [('VERTEX_AI_PROJECT_ID', 'dsi-tifin-harness'), ('VERTEX_AI_LOCATION', 'global'), ('VERTEX_MODEL', 'gemini-3.5-flash-lite')]})
print('This makes 24 paid Vertex requests: 12 answers and 12 post-task lessons.')

In [ ]:
environment = os.environ.copy()
environment.setdefault('VERTEX_AI_PROJECT_ID', 'dsi-tifin-harness')
environment.setdefault('VERTEX_AI_LOCATION', 'global')
environment.setdefault('VERTEX_MODEL', 'gemini-3.5-flash-lite')
run = subprocess.run(
    [sys.executable, str(RUNNER_PATH), str(MANIFEST_PATH), str(RESULTS_PATH), str(MEMORY_PATH)],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=False,
)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('Structured-lesson run stopped. Completed records and stored lessons were preserved.')

In [ ]:
results = [json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line]
assert len(results) == 12, f'Expected 12 completed records, found {len(results)}'
retrieval_tasks = [record for record in results if record['turn'] in (2, 3)]
print(f'Completed structured-lesson records: {len(results)} / 12')
print(f'Retrieval calls: {len(retrieval_tasks)} / 8 expected')
print(f'Retrievals marked relevant: {sum(record["retrieval_relevant"] is True for record in retrieval_tasks)} / {len(retrieval_tasks)}')
for record in retrieval_tasks:
    print(record['task_id'], '| ids:', record['retrieved_memory_ids'], '| overhead:', record['retrieval_overhead_ms'], 'ms')